# TRACE interactive walkthrough

This notebook loads precomputed video-window embeddings, prepares activity and concept sequences, trains a small TRACE model, and lets you intervene on future concept nodes or activity beliefs. Run cells from top to bottom.


In [ ]:
# Run this notebook from the repository, or open it in Jupyter and run cells in order.
# Before running: set TRACE_DATASET_ROOT and TRACE_EMBEDDING_ROOT in your shell or edit the two variables below.
# Example: export TRACE_DATASET_ROOT=/path/to/datasets and export TRACE_EMBEDDING_ROOT=/path/to/embeddings
# Set USE_CUDA=True below to request GPU compute; False forces CPU.
# Figures are saved under runs/notebook_demo/<RUN_TAG>/figures/ (ignored by Git).
# Use a new RUN_TAG to train another model and keep its figures separate.

from pathlib import Path
import os
import sys
import torch

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (cwd, *cwd.parents) if (p / 'train_models.py').is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError('Open this notebook from inside the TRACE-CBM repository.')
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Either edit these strings or set the matching environment variables before opening Jupyter.
DATASET_ROOT = Path(os.environ.get('TRACE_DATASET_ROOT', '/path/to/datasets')).expanduser()
EMBEDDING_ROOT = Path(os.environ.get('TRACE_EMBEDDING_ROOT', '/path/to/embeddings')).expanduser()
if str(DATASET_ROOT).startswith('/path/to/') or str(EMBEDDING_ROOT).startswith('/path/to/'):
    raise RuntimeError(
        'Set TRACE_DATASET_ROOT and TRACE_EMBEDDING_ROOT, or replace the two /path/to/... values in this cell.'
    )
os.environ['TRACE_DATASET_ROOT'] = str(DATASET_ROOT)
os.environ['TRACE_EMBEDDING_ROOT'] = str(EMBEDDING_ROOT)
# CPU is the safe first-run default in this environment. Set True to use CUDA when available.
USE_CUDA = False
DEVICE = 'cuda' if USE_CUDA and torch.cuda.is_available() else 'cpu'
if USE_CUDA and DEVICE == 'cpu':
    print('CUDA was requested but is unavailable; using CPU instead.')

RUN_TAG = 'run_02'
RUN_DIR = REPO_ROOT / 'runs' / 'notebook_demo' / RUN_TAG
FIGURES_DIR = RUN_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATASET = 'Breakfast'
TEST_SPLIT = 'official_s1'
WINDOW_SIZE = 32
BACKBONE = 'pe-l14'
CONCEPT_SET = 'breakfast_iterative_qwen35_pe_l14_128_v1'
EMBEDDING_PATH = EMBEDDING_ROOT / 'Breakfast' / 'True_32_clip_pe-l14.pkl'

print('Repository:', REPO_ROOT)
print('Compute device:', DEVICE)
print('Dataset files:', DATASET_ROOT)
print('Precomputed embeddings:', EMBEDDING_PATH)
print('Notebook figures:', FIGURES_DIR)


## 1. Check the prepared inputs

The embedding artifact stores one feature vector per video window, with paths and timestamps used for the optional preview. This notebook uses existing embeddings.


In [ ]:
from utils.intervention_notebook import (
    InterventionNotebookConfig,
    load_or_train_workspace,
    select_instance,
    run_instance,
    plot_instance_overview,
    plot_forecast_timeline,
    plot_concept_scores,
    intervene_on_activity,
    intervene_on_future_concept,
    plot_intervention_delta,
)
from utils.graph_concept_ui import forward_outputs_batched_interventions
from utils.intervention_notebook import _forecast_logits_by_horizon

if not EMBEDDING_PATH.is_file():
    raise FileNotFoundError(
        f'Prepared embeddings not found at {EMBEDDING_PATH}.\n'
        'Create them with the PE-L14 command in docs/DATASETS.md, or edit EMBEDDING_PATH above.'
    )
if not (DATASET_ROOT / 'Breakfast').is_dir():
    print('Note: the original video directory is not under DATASET_ROOT; prediction plots will still work, but video previews may be unavailable.')

config = InterventionNotebookConfig(
    dataset=DATASET,
    test_split=TEST_SPLIT,
    window_size=WINDOW_SIZE,
    concept_set=CONCEPT_SET,
    backbone=BACKBONE,
    data_root=EMBEDDING_ROOT,
    dataset_root=DATASET_ROOT,
    embedding_path=EMBEDDING_PATH,
    generate_embeddings=False,  # Avoid accidentally starting a long embedding job.
    output_dir=RUN_DIR / 'checkpoints',
    train_if_missing=True,
    base_method='trace',
    device=DEVICE,
    num_epochs=5,  # Short tutorial run; increase this for a longer experiment.
    patience=3,
    batch_size=64,
    seed=42,
    wandb_mode='disabled',
    # Enable the activity-to-concept feedback path so activity-belief edits can be ranked.
    model_hparams={
        'st_activity_feedback_mode': 'sparse_label_to_concept',
        'st_activity_feedback_top_k': 40,
        'st_activity_feedback_gate_init': 0.0,
        'st_activity_feedback_history_steps': 3,
        'st_activity_feedback_history_gate_init': 0.0,
    },
)

print('Device:', config.device)
print('Checkpoint folder:', config.output_dir)


## 2. Prepare the sequences and train TRACE

The next cell loads embeddings, aligns activity labels, computes concept activations, and trains TRACE only when the selected run folder has no checkpoint. Reusing the same `RUN_TAG` reloads its checkpoint.


In [ ]:
workspace = load_or_train_workspace(config)

print(f'Method: {workspace.base_method}')
print(f'Concepts: {len(workspace.concept_names)}')
print(f'Activity classes: {len(workspace.activity_names)}')
for split_name in ('train', 'val', 'test'):
    split = workspace.preprocessed_data[split_name]
    print(f"{split_name}: {len(split['lengths'])} videos, concepts={split['concepts'].shape}, labels={split['activity_labels'].shape}")
print('Checkpoint:', RUN_DIR / 'checkpoints' / 'latest' / 'model.pt')


## 3. Inspect an example

Change `VIDEO_INDEX` or `TIMESTEP` to inspect another test example. TRACE uses its observation context to form the current prediction; this walkthrough intervenes on future concept nodes or activity beliefs at H1–H3. Figures are written to `FIGURES_DIR`.


In [ ]:
VIDEO_INDEX = 1
TIMESTEP = 20  # None selects a timestep with room for all configured forecast horizons.
example = select_instance(workspace, split='test', video_index=VIDEO_INDEX, timestep=TIMESTEP)
summary = run_instance(workspace, example)
print('Video:', example['video_id'])
print('Source path:', example['video_path'])
print('Observed timestep:', example['timestep'])
print('Current activity: true =', summary['activity_true_label'], '| predicted =', summary['activity_pred_label'])
for horizon, forecast in summary['forecasts'].items():
    print(f"H{horizon}: true = {forecast['true_label']} | predicted = {forecast['pred_label']} | confidence = {forecast['probs'].max():.2f}")

import matplotlib.pyplot as plt

overview_figure = plot_instance_overview(workspace, example)
overview_figure.savefig(FIGURES_DIR / 'forecast_overview.png', dpi=160, bbox_inches='tight')
timeline_figure = plot_forecast_timeline(workspace, example)
timeline_figure.savefig(FIGURES_DIR / 'forecast_timeline.png', dpi=160, bbox_inches='tight')
concept_figure = plot_concept_scores(workspace, example, top_k=12)
concept_figure.savefig(FIGURES_DIR / 'concept_scores.png', dpi=160, bbox_inches='tight')
print('Saved plots to:', FIGURES_DIR)


## 4. Preview the observed video windows

The grid shows one frame near the centre of each observed window. It uses the local source video path saved with the embeddings.


In [ ]:
import cv2
import numpy as np
from PIL import Image


def resolve_local_video_path(video_path):
    """Resolve the saved embedding path against this machine's dataset root."""
    source_path = Path(video_path).expanduser()
    if source_path.exists():
        return source_path

    # Embeddings can contain paths saved on the machine that created them, e.g.
    # ../Datasets/Breakfast/Video_data/... . Re-root at the local DATASET_ROOT.
    parts = source_path.parts
    dataset_name = DATASET
    dataset_index = next(
        (i for i, part in enumerate(parts) if part.casefold() == dataset_name.casefold()),
        None,
    )
    candidates = []
    if dataset_index is not None:
        candidates.append(DATASET_ROOT.joinpath(*parts[dataset_index:]))
    if parts and parts[0].casefold() in {'video_data', 'image_data'}:
        candidates.append(DATASET_ROOT / dataset_name / Path(*parts))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return source_path


def read_window_frame(video_path, timestep, span=None, fps=15.0):
    path = Path(video_path)
    seconds = float(sum(span) / 2) if span is not None else (timestep * WINDOW_SIZE + WINDOW_SIZE / 2) / max(float(fps), 1.0)
    if path.is_dir():
        image_files = sorted(p for p in path.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        if not image_files:
            return None
        frame_index = min(max(int(seconds * fps), 0), len(image_files) - 1)
        return np.asarray(Image.open(image_files[frame_index]).convert('RGB'))
    if not path.is_file():
        return None
    capture = cv2.VideoCapture(str(path))
    if not capture.isOpened():
        return None
    capture.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000.0)
    ok, frame = capture.read()
    capture.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else None


embedding_video_path = example['video_path']
local_video_path = resolve_local_video_path(embedding_video_path)
metadata = workspace.preprocessed_data.get('metadata', {})
spans_by_video = metadata.get('video_window_spans', {})
video_meta_by_video = metadata.get('video_meta', {})
spans = spans_by_video.get(embedding_video_path, []) if isinstance(spans_by_video, dict) else []
video_meta = video_meta_by_video.get(embedding_video_path, {}) if isinstance(video_meta_by_video, dict) else {}
fps = float(video_meta.get('fps', 15.0) or 15.0)

real_steps = [example['history_start'] + i - example['history_offset'] for i, is_padding in enumerate(example['key_padding_mask']) if not is_padding]
frames = [(step, read_window_frame(local_video_path, step, spans[step] if step < len(spans) else None, fps)) for step in real_steps]
frames = [(step, frame) for step, frame in frames if frame is not None]
if frames:
    fig, axes = plt.subplots(1, len(frames), figsize=(3.2 * len(frames), 3.2), squeeze=False)
    for ax, (step, frame) in zip(axes[0], frames):
        ax.imshow(frame)
        ax.set_title(f'Observed window t={step}')
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'observed_video_windows.png', dpi=160, bbox_inches='tight')
    plt.show()
else:
    print('Could not read frames.')
    print('Embedding metadata path:', embedding_video_path)
    print('Resolved local path:', local_video_path)
    print('Check that the licensed video/image files exist below DATASET_ROOT:', DATASET_ROOT)


## 5. Rank future activity-belief edits by model response

This cell evaluates activity beliefs set to 0.9 for the H1–H3 rollout steps. It ranks them by their effect on the model's baseline top H3 predicted-class probability. The ranking uses no future ground-truth label.


In [ ]:
import pandas as pd
from IPython.display import display

FORECAST_HORIZON = max(summary['forecasts'])
BASELINE_FORECAST = summary['forecasts'][FORECAST_HORIZON]
TARGET_CLASS = int(BASELINE_FORECAST['pred_idx'])
TARGET_NAME = workspace.activity_names[TARGET_CLASS]
BASELINE_TARGET_PROB = float(BASELINE_FORECAST['probs'][TARGET_CLASS])


def target_probability_after_edits(payloads, chunk_size=16):
    """Return the target-class forecast probability after each batched edit."""
    probabilities = []
    for start in range(0, len(payloads), chunk_size):
        outputs = forward_outputs_batched_interventions(
            workspace,
            example,
            payloads[start:start + chunk_size],
        )
        logits = _forecast_logits_by_horizon(workspace, outputs)[FORECAST_HORIZON][:, -1, :]
        probabilities.extend(
            torch.softmax(logits, dim=-1)[:, TARGET_CLASS].detach().cpu().tolist()
        )
    return probabilities


print(
    f"Ranking edits by their effect on the baseline top H{FORECAST_HORIZON} "
    f"prediction '{TARGET_NAME}' (baseline probability {BASELINE_TARGET_PROB:.3f})."
)

# Activity beliefs propagate through the model's activity-to-concept feedback path.
feedback_enabled = callable(getattr(workspace.model, '_activity_feedback_enabled', None)) and bool(workspace.model._activity_feedback_enabled())
if feedback_enabled:
    activity_payloads, activity_descriptions = [], []
    for feedback_step in range(FORECAST_HORIZON):
        for class_idx, activity_name in enumerate(workspace.activity_names):
            activity_payloads.append({
                'mode': 'input',
                'items': [{
                    'item_type': 'activity',
                    'step': feedback_step,
                    'class_idx': class_idx,
                    'probability': 0.9,
                }],
            })
            activity_descriptions.append((feedback_step, activity_name))
    activity_after = target_probability_after_edits(activity_payloads)
    activity_rows = []
    for (feedback_step, activity_name), after_probability in zip(activity_descriptions, activity_after):
        activity_rows.append({
            'Used for forecast': f'H{feedback_step + 1}',
            'Activity belief set to 0.9': activity_name,
            'Target probability after': after_probability,
            'Change (percentage points)': 100.0 * (after_probability - BASELINE_TARGET_PROB),
        })
    activity_ranking = pd.DataFrame(activity_rows)
    activity_ranking['Absolute change (pp)'] = activity_ranking['Change (percentage points)'].abs()
    activity_ranking = activity_ranking.sort_values('Absolute change (pp)', ascending=False).head(10)
    print('\nStrongest future activity-belief edits:')
    display(activity_ranking.drop(columns='Absolute change (pp)').round(2).reset_index(drop=True))
else:
    print('\nThis checkpoint has activity-to-concept feedback disabled, so activity-belief edits cannot be ranked.')


## 6. Try a simple future concept-node intervention

The example raises one predicted concept node by 0.25 at H3, then shows its H3 forecast probability change. This is a future model-state intervention; it does not edit the video.


In [ ]:
# Raise one concept node in the final H3 predicted state.
concept_to_edit = workspace.concept_names[0]
intervention_horizon = max(summary['forecasts'])
intervention = intervene_on_future_concept(
    workspace, example, concept=concept_to_edit, horizon=intervention_horizon, delta=0.25
)
intervention_figure = plot_intervention_delta(workspace, intervention, horizon=intervention_horizon)
intervention_figure.savefig(FIGURES_DIR / 'future_concept_intervention_delta.png', dpi=160, bbox_inches='tight')
print(f"Raised H{intervention_horizon} concept node '{concept_to_edit}' by 0.25.")
print('Saved intervention plot to:', FIGURES_DIR / 'future_concept_intervention_delta.png')


## 7. Intervene on future concept nodes or activity beliefs

Choose a concept node or activity belief and a future horizon H1, H2, or H3. Concept-node edits change the predicted concept state at that horizon; activity-belief edits set the belief used to produce that forecast. Each click replaces one image widget with one probability-change plot for the selected horizon.


In [ ]:
import html
import io
import traceback
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

# Remove controls from a previous execution of this cell. This prevents old
# buttons and their callbacks from staying live in the current kernel.
for _widget in globals().get('_trace_intervention_controls', []):
    _widget.close()

concept_options = [(name, i) for i, name in enumerate(workspace.concept_names)]
activity_options = [(name, i) for i, name in enumerate(workspace.activity_names)]
concept_picker = widgets.Dropdown(options=concept_options, value=0, description='Concept node')
strongest_activity_idx = 0
if 'activity_ranking' in globals() and not activity_ranking.empty:
    strongest_activity_name = str(activity_ranking.iloc[0]['Activity belief set to 0.9'])
    strongest_activity_idx = workspace.activity_names.index(strongest_activity_name)
activity_picker = widgets.Dropdown(options=activity_options, value=strongest_activity_idx, description='Activity belief')
horizon_options = [(f'H{horizon} (t+{horizon})', horizon) for horizon in sorted(summary['forecasts'])]
horizon_picker = widgets.Dropdown(options=horizon_options, value=max(summary['forecasts']), description='Future step')
edit_kind = widgets.ToggleButtons(options=['Concept node', 'Activity belief'], value='Concept node', description='Edit type')
node_amount = widgets.FloatSlider(value=0.25, min=-1.0, max=1.0, step=0.05, description='Node change')
activity_probability = widgets.FloatSlider(value=0.9, min=0.1, max=0.99, step=0.05, description='Belief')
run_button = widgets.Button(description='Run intervention', button_style='primary')
status = widgets.HTML(value='<b>Status:</b> ready')
result_summary = widgets.HTML()
plot_image = widgets.Image(format='png')

def sync_edit_controls(_=None):
    editing_concept = edit_kind.value == 'Concept node'
    concept_picker.disabled = not editing_concept
    node_amount.disabled = not editing_concept
    activity_picker.disabled = editing_concept
    activity_probability.disabled = editing_concept

edit_kind.observe(sync_edit_controls, names='value')
sync_edit_controls()

# Reload the helper so a source fix can be used by rerunning this widget cell.
import importlib
import utils.intervention_notebook as intervention_notebook_module
intervention_notebook_module = importlib.reload(intervention_notebook_module)
intervene_on_activity = intervention_notebook_module.intervene_on_activity
intervene_on_future_concept = intervention_notebook_module.intervene_on_future_concept
plot_intervention_delta = intervention_notebook_module.plot_intervention_delta
def on_run_intervention(button):
    try:
        status.value = '<b>Status:</b> running…'
        intervention_horizon = int(horizon_picker.value)
        if edit_kind.value == 'Concept node':
            result = intervene_on_future_concept(
                workspace, example, concept=concept_picker.value,
                horizon=intervention_horizon, delta=node_amount.value,
            )
            status_text = f"concept node '{result['concept_name']}' edited at H{intervention_horizon}"
            summary_lines = [
                f"Changed predicted H{intervention_horizon} concept node <b>{html.escape(result['concept_name'])}</b> by {node_amount.value:+.2f}.",
            ]
        else:
            result = intervene_on_activity(
                workspace, example, activity=activity_picker.value,
                step=intervention_horizon - 1, probability=activity_probability.value,
            )
            status_text = f"activity belief '{result['activity']}' set for H{intervention_horizon}"
            summary_lines = [
                f"Set the H{intervention_horizon} rollout belief for <b>{html.escape(result['activity'])}</b> to {activity_probability.value:.2f}.",
            ]
        before, after = result['before'], result['after']
        for horizon in sorted(before['forecasts']):
            before_row, after_row = before['forecasts'][horizon], after['forecasts'][horizon]
            summary_lines.append(
                f"+{horizon}: {html.escape(before_row['pred_label'])} → {html.escape(after_row['pred_label'])}"
            )
        result_summary.value = '<br>'.join(summary_lines)
        figure = plot_intervention_delta(workspace, result, horizon=intervention_horizon, show=False)
        image_buffer = io.BytesIO()
        figure.savefig(image_buffer, format='png', dpi=130, bbox_inches='tight')
        plt.close(figure)
        plot_image.value = image_buffer.getvalue()
        status.value = '<b>Status:</b> complete — ' + html.escape(status_text) + '.'
    except Exception as error:
        status.value = '<b>Status:</b> failed — see the error below.'
        result_summary.value = (
            f"<b>{html.escape(type(error).__name__)}</b>: {html.escape(str(error))}"
            f"<pre>{html.escape(traceback.format_exc())}</pre>"
        )
run_button.on_click(on_run_intervention)
_trace_intervention_controls = [
    edit_kind, concept_picker, activity_picker, horizon_picker, node_amount, activity_probability,
    run_button, status, result_summary, plot_image,
]
print('Choose a future concept node or activity belief at H1–H3, then click Run intervention.')
display(widgets.VBox([
    edit_kind,
    concept_picker,
    activity_picker,
    horizon_picker,
    node_amount,
    activity_probability,
    run_button,
    status,
    result_summary,
    plot_image,
]))


## Outputs

Plots are saved in `runs/notebook_demo/<RUN_TAG>/figures/`; the checkpoint is stored in `runs/notebook_demo/<RUN_TAG>/checkpoints/latest/model.pt`. Generated outputs are ignored by Git.
